# 面试问题：GraphRAG 怎样结合 Local Search、Community Summary 和 Provenance？

可以直接复述的回答是：第一，从文本抽取实体和有类型的关系边。第二，每条边必须保留来源片段。第三，Local Search 从问题实体出发做受限 BFS，回答具体多跳关系。第四，Global Search 先按社区汇总，再比较跨社区主题。第五，高度中心节点和低可信边会制造错误捷径。第六，应输出路径、社区、引用边和证据覆盖率。下面用供应链风险图实现。

## 真实案例：两个产品线的供应商延迟影响分析

图包含 10 个实体、10 条可信关系和 1 条低可信通用公司边。实体覆盖供应商、组件、工厂、产品与事故。五个问题既有具体影响路径，也有跨社区全局风险。所有名称和来源编号均为脱敏教学数据。

In [1]:
nodes = {  # 定义十个具有类型和名称的供应链实体
    "S-A": {"type": "supplier", "name": "供应商 A"},  # 提供芯片 X 的供应商
    "C-X": {"type": "component", "name": "芯片 X"},  # Alpha 产品关键组件
    "F-SH": {"type": "factory", "name": "上海工厂"},  # 组装 Alpha 的工厂
    "P-A": {"type": "product", "name": "产品 Alpha"},  # 第一条产品线
    "I-1": {"type": "incident", "name": "港口延迟"},  # 第一社区风险事件
    "S-B": {"type": "supplier", "name": "供应商 B"},  # 提供电池 Y 的供应商
    "C-Y": {"type": "component", "name": "电池 Y"},  # Beta 产品关键组件
    "F-CD": {"type": "factory", "name": "成都工厂"},  # 组装 Beta 的工厂
    "P-B": {"type": "product", "name": "产品 Beta"},  # 第二条产品线
    "I-2": {"type": "incident", "name": "质检失败"},  # 第二社区风险事件
}  # 结束十个图节点
edges = [  # 定义带类型、强度和来源的可信关系边
    {"source": "S-A", "target": "C-X", "type": "supplies", "strength": 1.0, "citation": "DOC-01", "trusted": True},  # 供应商 A 提供芯片 X
    {"source": "C-X", "target": "F-SH", "type": "shipped_to", "strength": 0.9, "citation": "DOC-02", "trusted": True},  # 芯片运往上海工厂
    {"source": "F-SH", "target": "P-A", "type": "assembles", "strength": 1.0, "citation": "DOC-03", "trusted": True},  # 上海工厂组装 Alpha
    {"source": "I-1", "target": "S-A", "type": "delays", "strength": 0.9, "citation": "NEWS-01", "trusted": True},  # 港口延迟影响供应商 A
    {"source": "S-B", "target": "C-Y", "type": "supplies", "strength": 1.0, "citation": "DOC-04", "trusted": True},  # 供应商 B 提供电池 Y
    {"source": "C-Y", "target": "F-CD", "type": "shipped_to", "strength": 0.9, "citation": "DOC-05", "trusted": True},  # 电池运往成都工厂
    {"source": "F-CD", "target": "P-B", "type": "assembles", "strength": 1.0, "citation": "DOC-06", "trusted": True},  # 成都工厂组装 Beta
    {"source": "I-2", "target": "C-Y", "type": "blocks", "strength": 0.9, "citation": "QA-01", "trusted": True},  # 质检失败阻断电池 Y
    {"source": "F-SH", "target": "F-CD", "type": "backup_route", "strength": 0.4, "citation": "PLAN-01", "trusted": True},  # 两工厂之间存在弱备援关系
    {"source": "S-A", "target": "S-B", "type": "same_group", "strength": 0.3, "citation": "REG-01", "trusted": True},  # 两供应商同集团但非供货关系
]  # 结束可信供应链关系
questions = [  # 定义五个 Local 或 Global GraphRAG 问题
    {"id": "GQ-01", "text": "港口延迟会影响哪个产品？", "mode": "local", "start": "I-1", "target_type": "product", "gold": {"NEWS-01", "DOC-01", "DOC-02", "DOC-03"}},  # 四跳影响路径
    {"id": "GQ-02", "text": "供应商 B 最终服务哪个产品？", "mode": "local", "start": "S-B", "target_type": "product", "gold": {"DOC-04", "DOC-05", "DOC-06"}},  # 三跳供货路径
    {"id": "GQ-03", "text": "质检失败在哪条产品线？", "mode": "local", "start": "I-2", "target_type": "product", "gold": {"QA-01", "DOC-05", "DOC-06"}},  # 事故到产品路径
    {"id": "GQ-04", "text": "上海工厂组装什么产品？", "mode": "local", "start": "F-SH", "target_type": "product", "gold": {"DOC-03"}},  # 单跳局部问题
    {"id": "GQ-05", "text": "两个产品社区各自的主要风险是什么？", "mode": "global", "start": None, "target_type": "incident", "gold": {"NEWS-01", "QA-01"}},  # 跨社区风险总结
]  # 结束五个图问题
print("图输入：nodes=", len(nodes), "edges=", len(edges))  # 展示 GraphRAG 的真实规模
for edge in edges:  # 逐边输出类型和来源
    print(f"{edge['source']} -[{edge['type']}]-> {edge['target']} | strength={edge['strength']} | {edge['citation']}")  # 呈现可重放关系证据
print("问题集：", [(item["id"], item["mode"], item["text"]) for item in questions])  # 展示五个 Local/Global 查询


图输入：nodes= 10 edges= 10
S-A -[supplies]-> C-X | strength=1.0 | DOC-01
C-X -[shipped_to]-> F-SH | strength=0.9 | DOC-02
F-SH -[assembles]-> P-A | strength=1.0 | DOC-03
I-1 -[delays]-> S-A | strength=0.9 | NEWS-01
S-B -[supplies]-> C-Y | strength=1.0 | DOC-04
C-Y -[shipped_to]-> F-CD | strength=0.9 | DOC-05
F-CD -[assembles]-> P-B | strength=1.0 | DOC-06
I-2 -[blocks]-> C-Y | strength=0.9 | QA-01
F-SH -[backup_route]-> F-CD | strength=0.4 | PLAN-01
S-A -[same_group]-> S-B | strength=0.3 | REG-01
问题集： [('GQ-01', 'local', '港口延迟会影响哪个产品？'), ('GQ-02', 'local', '供应商 B 最终服务哪个产品？'), ('GQ-03', 'local', '质检失败在哪条产品线？'), ('GQ-04', 'local', '上海工厂组装什么产品？'), ('GQ-05', 'global', '两个产品社区各自的主要风险是什么？')]


## Baseline / 基线：把关系句子当普通文本 Top-2

关键词 Top-2 能找到起点附近的句子，却无法覆盖三到四跳路径。我们计算每题证据覆盖率作为同口径基线。

In [2]:
edge_chunks = [{"text": f"{nodes[edge['source']]['name']} {edge['type']} {nodes[edge['target']]['name']}", "citation": edge["citation"]} for edge in edges]  # 把每条边退化为独立文本块
def character_terms(text):  # 提取中文字符和字母数字作为透明关键词
    return {character.lower() for character in text if character.strip()}  # 返回去重字符集合
def baseline_retrieve(question, top_k=2):  # 实现无图结构的文本重合检索
    query_terms = character_terms(question["text"])  # 获取问题字符特征
    scored = [(len(query_terms & character_terms(chunk["text"])), chunk["citation"]) for chunk in edge_chunks]  # 计算每条边文本的重合数
    return [citation for _, citation in sorted(scored, key=lambda item: (-item[0], item[1]))[:top_k]]  # 返回两个最高分来源
print("文本 Baseline：id | top2 | evidence_coverage")  # 输出五题的证据覆盖
baseline_coverages = []  # 收集无图检索覆盖率
for question in questions:  # 逐问题运行 Top-2 文本检索
    citations = set(baseline_retrieve(question))  # 获取基线两个来源
    coverage = len(citations & question["gold"]) / len(question["gold"])  # 计算人工路径证据覆盖率
    baseline_coverages.append(coverage)  # 保存基线指标
    print(f"{question['id']} | {sorted(citations)} | {coverage:.0%}")  # 展示多跳问题的缺失边


文本 Baseline：id | top2 | evidence_coverage
GQ-01 | ['DOC-03', 'NEWS-01'] | 50%
GQ-02 | ['DOC-04', 'REG-01'] | 33%
GQ-03 | ['DOC-03', 'QA-01'] | 33%
GQ-04 | ['DOC-02', 'DOC-03'] | 100%
GQ-05 | ['DOC-03', 'DOC-06'] | 0%


## 核心实现：强关系社区、受限 BFS 与 Provenance 路径

先用强度 ≥0.8 的可信边构建社区。Local Search 在允许关系上 BFS 并保存父边；Global Search 对每个社区收集 incident 与来源。

In [3]:
from collections import deque  # 使用双端队列实现受限广度优先搜索
strong_adjacency = {node_id: set() for node_id in nodes}  # 初始化强关系无向图用于社区发现
for edge in edges:  # 遍历十条可信关系
    if edge["trusted"] and edge["strength"] >= 0.8:  # 只用高置信业务关系形成社区
        strong_adjacency[edge["source"]].add(edge["target"])  # 添加正向社区连接
        strong_adjacency[edge["target"]].add(edge["source"])  # 添加反向社区连接
communities = []  # 收集强关系连通分量
unseen = set(nodes)  # 初始化尚未分配社区的节点
while unseen:  # 持续发现所有连通分量
    root = min(unseen)  # 使用稳定顺序选择社区起点
    queue = deque([root])  # 初始化当前社区 BFS 队列
    component = set()  # 收集当前社区节点
    while queue:  # 遍历所有强关系可达节点
        current = queue.popleft()  # 取出下一个待访问节点
        if current in component:  # 已访问节点无需重复处理
            continue  # 跳过重复队列项
        component.add(current)  # 把当前节点加入社区
        queue.extend(sorted(strong_adjacency[current] - component))  # 加入尚未访问的强关系邻居
    communities.append(component)  # 保存完整社区
    unseen -= component  # 从待分配集合移除当前社区
allowed_relation_types = {"supplies", "shipped_to", "assembles", "delays", "blocks"}  # 定义 Local Search 可用于因果路径的关系
def local_path(start, target_type, max_depth=4):  # 查找起点到目标类型的最短可信路径
    queue = deque([(start, [])])  # 队列保存当前节点和已走边列表
    visited = {start}  # 防止供应链环导致重复扩展
    while queue:  # 按路径长度逐层搜索
        current, path = queue.popleft()  # 读取当前实体和 provenance 路径
        if current != start and nodes[current]["type"] == target_type:  # 首次命中目标类型即返回最短路径
            return current, path  # 返回目标节点和完整来源边
        if len(path) >= max_depth:  # 超过教学跳数预算时停止扩展
            continue  # 保持查询成本有界
        for edge in edges:  # 检查当前节点可沿用的有向可信边
            if edge["source"] == current and edge["trusted"] and edge["type"] in allowed_relation_types and edge["target"] not in visited:  # 过滤弱关系和重复节点
                visited.add(edge["target"])  # 标记目标节点已进入搜索
                queue.append((edge["target"], path + [edge]))  # 复制路径并追加当前来源边
    return None, []  # 没有可信路径时返回空结果
target_q1, path_q1 = local_path("I-1", "product")  # 搜索港口延迟到产品的四跳路径
print("强关系社区：", [sorted(component) for component in communities])  # 展示两个产品线社区结构
print("GQ-01 Local Path：")  # 输出核心多跳检索轨迹
for edge in path_q1:  # 逐边展示事故到产品的路径
    print(f"{nodes[edge['source']]['name']} -[{edge['type']}]-> {nodes[edge['target']]['name']} [{edge['citation']}]")  # 同时展示关系和引用来源
print("GQ-01 目标产品：", nodes[target_q1]["name"])  # 展示 Local Search 最终答案


强关系社区： [['C-X', 'F-SH', 'I-1', 'P-A', 'S-A'], ['C-Y', 'F-CD', 'I-2', 'P-B', 'S-B']]
GQ-01 Local Path：
港口延迟 -[delays]-> 供应商 A [NEWS-01]
供应商 A -[supplies]-> 芯片 X [DOC-01]
芯片 X -[shipped_to]-> 上海工厂 [DOC-02]
上海工厂 -[assembles]-> 产品 Alpha [DOC-03]
GQ-01 目标产品： 产品 Alpha


## 失败案例与修正：通用公司 Hub 制造错误捷径

低可信网页声称所有实体都属于“同一公司”，若把无类型 Hub 边加入 BFS，供应商 A 可两跳连接产品 Beta。修正只接受可信来源和业务关系类型。

In [4]:
hub_edges = [{"source": "S-A", "target": "P-B", "type": "related_to", "strength": 0.2, "citation": "WEB-UNTRUSTED", "trusted": False}]  # 构造低可信通用关联捷径
all_edges = edges + hub_edges  # 把错误 Hub 关系加入未治理图
unsafe_shortcut = next((edge for edge in all_edges if edge["source"] == "S-A" and edge["target"] == "P-B"), None)  # 模拟不筛类型的检索命中
safe_shortcut = next((edge for edge in all_edges if edge["source"] == "S-A" and edge["target"] == "P-B" and edge["trusted"] and edge["type"] in allowed_relation_types), None)  # 应用可信和类型门禁
safe_target, safe_path = local_path("S-A", "product")  # 在原可信图中重新搜索供应商 A 的产品路径
print("修正前捷径：", unsafe_shortcut)  # 展示错误关系、来源和低强度
print("修正后捷径：", safe_shortcut)  # 展示低可信 related_to 不进入搜索
print("可信路径目标：", nodes[safe_target]["name"], "citations=", [edge["citation"] for edge in safe_path])  # 展示供应商 A 正确连接 Alpha


修正前捷径： {'source': 'S-A', 'target': 'P-B', 'type': 'related_to', 'strength': 0.2, 'citation': 'WEB-UNTRUSTED', 'trusted': False}
修正后捷径： None
可信路径目标： 产品 Alpha citations= ['DOC-01', 'DOC-02', 'DOC-03']


## 结果表：五题 Local/Global 证据覆盖

In [5]:
def global_risks():  # 为每个强关系社区生成风险摘要与来源
    summaries = []  # 收集社区节点、事故和引用
    for component in communities:  # 逐社区寻找 incident 节点及其相邻来源
        incidents = [node_id for node_id in component if nodes[node_id]["type"] == "incident"]  # 获取当前社区事故实体
        citations = {edge["citation"] for edge in edges if edge["trusted"] and (edge["source"] in incidents or edge["target"] in incidents)}  # 收集事故相关边引用
        if incidents:  # 只输出包含风险事件的业务社区
            summaries.append({"community": sorted(component), "incidents": [nodes[node_id]["name"] for node_id in incidents], "citations": citations})  # 保存全局摘要证据
    return summaries  # 返回跨社区风险列表
graph_coverages = []  # 收集 GraphRAG 证据覆盖率
print("id | mode | answer | citations | coverage")  # 输出逐问题图检索结果
for question in questions:  # 对四个 Local 和一个 Global 问题执行
    if question["mode"] == "local":  # Local Search 返回目标实体和路径
        target, path = local_path(question["start"], question["target_type"])  # 搜索最短可信关系路径
        answer = nodes[target]["name"] if target else "not_found"  # 转换目标节点为可读答案
        citations = {edge["citation"] for edge in path}  # 收集路径上所有来源
    else:  # Global Search 汇总两个强关系社区
        summaries = global_risks()  # 获取社区级事故摘要
        answer = "；".join(f"{','.join(summary['incidents'])}" for summary in summaries)  # 组合跨社区风险名称
        citations = set().union(*(summary["citations"] for summary in summaries))  # 汇总所有事故来源
    coverage = len(citations & question["gold"]) / len(question["gold"])  # 计算人工证据覆盖率
    graph_coverages.append(coverage)  # 保存 GraphRAG 指标
    print(f"{question['id']} | {question['mode']} | {answer} | {sorted(citations)} | {coverage:.0%}")  # 展示答案、引用和覆盖率
mean_baseline_coverage = sum(baseline_coverages) / len(baseline_coverages)  # 计算文本 Top-2 平均覆盖率
mean_graph_coverage = sum(graph_coverages) / len(graph_coverages)  # 计算 Local/Global GraphRAG 平均覆盖率
print(f"平均证据覆盖率：text_top2={mean_baseline_coverage:.1%}，GraphRAG={mean_graph_coverage:.1%}")  # 输出同一五题对照


id | mode | answer | citations | coverage
GQ-01 | local | 产品 Alpha | ['DOC-01', 'DOC-02', 'DOC-03', 'NEWS-01'] | 100%
GQ-02 | local | 产品 Beta | ['DOC-04', 'DOC-05', 'DOC-06'] | 100%
GQ-03 | local | 产品 Beta | ['DOC-05', 'DOC-06', 'QA-01'] | 100%
GQ-04 | local | 产品 Alpha | ['DOC-03'] | 100%
GQ-05 | global | 港口延迟；质检失败 | ['NEWS-01', 'QA-01'] | 100%
平均证据覆盖率：text_top2=43.3%，GraphRAG=100.0%


## 结果解读

GQ-01 从港口延迟沿 delays、supplies、shipped_to、assembles 四条有来源边抵达产品 Alpha，普通 Top-2 无法覆盖完整链。社区发现把两个产品线按强业务关系分开，Global Search 各自引用 NEWS-01 与 QA-01。低可信 Hub 被拒绝，说明图结构本身也需要来源和关系类型治理。

## 生产边界

真实 GraphRAG 需要实体消歧、关系抽取置信度、动态图更新、社区算法、社区摘要版本和图权限。BFS 跳数、边类型与方向应针对问题规划，不能把相关关系解释为因果。来源删除或修订必须能沿边反向失效。本例使用人工图和小型连通分量。

## 最小回归测试

In [6]:
assert len(nodes) >= 5 and len(questions) >= 5  # 保证图和问题集具有真实多跳规模
assert target_q1 == "P-A" and len(path_q1) == 4  # 保证港口延迟通过四条关系定位 Alpha
assert {edge["citation"] for edge in path_q1} == questions[0]["gold"]  # 保证 Local Path 引用完整
assert unsafe_shortcut is not None and safe_shortcut is None  # 保证低可信 Hub 捷径被关系门禁修正
assert safe_target == "P-A"  # 保证供应商 A 不会被错误连接到 Beta
assert mean_graph_coverage > mean_baseline_coverage  # 保证 GraphRAG 在同一五题上提高证据覆盖率
